# Projeto (Seção 10) — solução comentada

Continua a solução comentada da seção 9. Enunciado em [`03_projeto`](03_projeto.ipynb).

Pedido: tirar o `@field_serializer` de datas do modelo e criar o tipo `CustomDate`. Além disso, corrijo a assimetria que a solução do
instrutor deixa: um tipo que **escreve** `YYYY/MM/DD` também precisa **ler** `YYYY/MM/DD`, senão o JSON gerado pelo modelo não volta
para o modelo.

## 1. O tipo `CustomDate`

In [1]:
import re
from datetime import date, datetime
from typing import Annotated, Any

from pydantic import BeforeValidator, PlainSerializer, TypeAdapter, ValidationError

DATE_FORMAT = "%Y/%m/%d"


SLASH_DATE = re.compile(r"\d{4}/\d{1,2}/\d{1,2}")


def parse_custom_date(value: Any) -> Any:
    """Aceita o formato de saída (YYYY/MM/DD); qualquer outra coisa segue para a validação padrão de `date`."""
    if isinstance(value, str) and SLASH_DATE.fullmatch(value.strip()):
        try:
            return datetime.strptime(value.strip(), DATE_FORMAT).date()
        except ValueError as ex:               # formato certo, data impossível (ex.: mês 13)
            raise ValueError(f"invalid date {value!r}: {ex}") from None
    return value                               # ex.: ISO "2023-06-01" -> o Pydantic resolve


def serialize_custom_date(value: date) -> str:
    return value.strftime(DATE_FORMAT)


CustomDate = Annotated[
    date,
    BeforeValidator(parse_custom_date),
    PlainSerializer(serialize_custom_date, return_type=str, when_used="json-unless-none"),
]

In [2]:
custom_date = TypeAdapter(CustomDate)
print(f"custom_date.validate_python('2023-06-01'): {custom_date.validate_python('2023-06-01')!r}")
print(f"custom_date.validate_python('2023/06/01'): {custom_date.validate_python('2023/06/01')!r}")
print(f"custom_date.dump_python(date(2023, 6, 1)): {custom_date.dump_python(date(2023, 6, 1))!r}")
print(f"custom_date.dump_json(date(2023, 6, 1)): {custom_date.dump_json(date(2023, 6, 1))}")
print(f"ida e volta pelo JSON: {custom_date.validate_json(custom_date.dump_json(date(2023, 6, 1)))!r}")
for bad in ["2023/13/01", "01/06/2023", "ontem"]:
    try:
        custom_date.validate_python(bad)
    except ValidationError as ex:
        print(f"{bad!r}: {ex.errors()[0]['msg']}")

custom_date.validate_python('2023-06-01'): datetime.date(2023, 6, 1)
custom_date.validate_python('2023/06/01'): datetime.date(2023, 6, 1)
custom_date.dump_python(date(2023, 6, 1)): datetime.date(2023, 6, 1)
custom_date.dump_json(date(2023, 6, 1)): b'"2023/06/01"'
ida e volta pelo JSON: datetime.date(2023, 6, 1)
'2023/13/01': Value error, invalid date '2023/13/01': time data '2023/13/01' does not match format '%Y/%m/%d'
'01/06/2023': Input should be a valid date or datetime, invalid character in year
'ontem': Input should be a valid date or datetime, input is too short


> 💡 `return_type=str` no `PlainSerializer` deixa explícito, para o schema JSON de serialização, que a saída é texto (o Pydantic
> também deduz isso da anotação `-> str` da função; aqui fica documentado no próprio tipo).

## 2. O modelo

In [3]:
from enum import Enum
from functools import cached_property
from typing import TypeVar
from uuid import UUID, uuid4
from pydantic import (UUID4, AfterValidator, AliasChoices, BaseModel, ConfigDict, Field, ValidationInfo,
                      computed_field, field_validator)
from pydantic.alias_generators import to_camel

countries = {
    "australia": ("Australia", "AUS"), "canada": ("Canada", "CAN"), "china": ("China", "CHN"),
    "france": ("France", "FRA"), "germany": ("Germany", "DEU"), "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"), "norway": ("Norway", "NOR"), "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"), "sanmarino": ("San Marino", "SMR"), "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"), "united kingdom": ("United Kingdom", "GBR"), "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"), "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"), "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"), "brazil": ("Brazil", "BRA"), "brasil": ("Brazil", "BRA"),
    # nomes canônicos também resolvem (necessário para reler o que o próprio modelo serializa)
    "united states of america": ("United States of America", "USA"),
}
VALID_COUNTRY_KEYS = sorted(countries)
country_code_lookup = {name: code for name, code in countries.values()}


def lookup_country(name: str) -> tuple[str, str]:
    key = " ".join(name.split()).casefold()
    try:
        return countries[key]
    except KeyError:
        raise ValueError(f"unknown country {name!r}; expected one of: {', '.join(VALID_COUNTRY_KEYS)}") from None


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


T = TypeVar("T")
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]
Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]


def make_alias(field_name: str) -> str:
    return to_camel(field_name).removesuffix("_")


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,
    )

    id_: UUID4 = Field(default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType
    is_electric: bool = Field(default=False, repr=False)
    manufactured_date: CustomDate = Field(
        validation_alias=AliasChoices("completionDate", "manufacturedDate"), ge=date(1980, 1, 1), repr=False)
    base_msrp_usd: float = Field(
        validation_alias=AliasChoices("msrpUSD", "baseMSRPUSD"), serialization_alias="baseMSRPUSD", repr=False)
    top_features: BoundedList[BoundedString] | None = Field(default=None, repr=False)
    vin: BoundedString = Field(repr=False)
    number_of_doors: int = Field(
        default=4, validation_alias=AliasChoices("doors", "numberOfDoors"), ge=2, le=4, multiple_of=2, repr=False)
    registration_country: Country | None = Field(default=None, repr=False)
    registration_date: CustomDate | None = Field(default=None, repr=False)
    license_plate: BoundedString | None = Field(default=None, repr=False)

    @computed_field(repr=False)
    @cached_property
    def registration_country_code(self) -> str | None:
        if self.registration_country is None:
            return None
        return country_code_lookup[self.registration_country]

    def __setattr__(self, name, value):
        super().__setattr__(name, value)
        if name == "registration_country":
            self.__dict__.pop("registration_country_code", None)

    @field_validator("manufactured_date")
    @classmethod
    def manufactured_not_after_registration(cls, value: date, info: ValidationInfo) -> date:
        registered = info.data.get("registration_date")
        if registered is not None and value > registered:
            raise ValueError(f"manufactured_date {value} is after registration_date {registered}")
        return value

    @field_validator("registration_date")
    @classmethod
    def registration_not_before_manufacture(cls, value: date | None, info: ValidationInfo) -> date | None:
        manufactured = info.data.get("manufactured_date")
        if value is not None and manufactured is not None and value < manufactured:
            raise ValueError(f"registration_date {value} is before manufactured_date {manufactured}")
        return value

O `@field_serializer` sumiu do modelo: a regra de formato vive no tipo.

Duas mudanças além do pedido, ambas a serviço da ida e volta pelo JSON:
- `AliasChoices` nos campos com nome de entrada diferente do de saída (`completionDate`/`manufacturedDate` etc.), como na solução da seção 4;
- o nome canônico `"united states of america"` passa a ser uma chave válida — o modelo serializa o nome canônico e precisa conseguir relê-lo.

## 3. Testes

In [4]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "us",
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

car = Automobile.model_validate(data)
print(f"car: {car!r}")
python_dump = car.model_dump(by_alias=True)
print(f"python_dump['manufacturedDate']: {python_dump['manufacturedDate']!r}")
print(f"python_dump['registrationDate']: {python_dump['registrationDate']!r}")
json_text = car.model_dump_json(by_alias=True)
print(f"json_text: {json_text}")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>)
python_dump['manufacturedDate']: datetime.date(2023, 1, 1)
python_dump['registrationDate']: datetime.date(2023, 6, 1)
json_text: {"id":"c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7","manufacturer":"BMW","seriesName":"M4 Competition xDrive","type":"Convertible","isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,"topFeatures":["6 cylinders","all-wheel drive","convertible"],"vin":"1234567890","numberOfDoors":2,"registrationCountry":"United States of America","registrationDate":"2023/06/01","licensePlate":"AAA-BBB","registrationCountryCode":"USA"}


In [5]:
import json

# datas: objetos date no dict, YYYY/MM/DD no JSON
assert python_dump["manufacturedDate"] == date(2023, 1, 1)
assert json.loads(json_text)["registrationDate"] == "2023/06/01"

# None continua null (json-unless-none) e não quebra o código do país
no_reg = {k: v for k, v in data.items() if k not in ("registrationDate", "registrationCountry")}
no_reg_json = json.loads(Automobile.model_validate(no_reg).model_dump_json(by_alias=True))
print(f"sem registro: registrationDate={no_reg_json['registrationDate']}, registrationCountryCode={no_reg_json['registrationCountryCode']}")

# ida e volta: o JSON do modelo volta para o modelo (campo calculado fica de fora: extra="forbid")
round_trip = Automobile.model_validate_json(car.model_dump_json(by_alias=True, exclude={"registration_country_code"}))
assert round_trip == car
print(f"round_trip == car: {round_trip == car}")

# validação cruzada continua valendo com datas no formato YYYY/MM/DD
try:
    Automobile.model_validate(data | {"registrationDate": "2022/12/31"})
except ValidationError as ex:
    print(f"registro antes da fabricação: {ex.errors()[0]['msg']}")

# schema de serialização: datas como string
props = Automobile.model_json_schema(mode="serialization", by_alias=True)["properties"]
print(f"props['manufacturedDate']: {props['manufacturedDate']}")
print("testes: ok")

sem registro: registrationDate=None, registrationCountryCode=None
round_trip == car: True
registro antes da fabricação: Value error, registration_date 2022-12-31 is before manufactured_date 2023-01-01
props['manufacturedDate']: {'title': 'Manufactureddate', 'type': 'string'}
testes: ok


## 4. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| `CustomDate` com `PlainSerializer(when_used="json-unless-none")` | ✅ | ✅ |
| `CustomDate` lê o próprio formato (`YYYY/MM/DD`) | ❌ — JSON gerado não volta | ✅ `BeforeValidator` |
| ida e volta do modelo pelo JSON | ❌ (data, aliases, nome do país, campo calculado) | ✅ com `exclude` do campo calculado |
| bugs herdados (país `None`, cache, `TypeError`) | presentes | corrigidos nas seções 8 e 9 |

> ⚠️ O `exclude={"registration_country_code"}` na ida e volta é necessário porque campos calculados entram na serialização e
> `extra="forbid"` os rejeita na entrada. Alternativa: um `@model_validator(mode="before")` que descarte `registrationCountryCode`
> — mais conveniente para o consumidor, mas esconde dados inconsistentes (código que não bate com o país).